<a href="https://colab.research.google.com/github/th100129/norajo/blob/main/KLUE%20GLiNER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets transformers torch tqdm
import os
import re
import json
import random
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

# =========================================================
# 0. 기본 설정
# =========================================================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

SAVE_DIR = "/content/klue_ner_globalpointer"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_NAME = "klue/roberta-base"
MAX_LENGTH = 128

LABELS = ["DT", "LC", "OG", "PS", "QT", "TI"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("SAVE_DIR:", SAVE_DIR)
print("LABELS:", LABELS)
# =========================================================
# 1. 태그 문장 파싱
# 예: "특히 <영동고속도로:LC> <강릉:LC> 방향 ..."
# -> text + entities(start, end, label, text)
# =========================================================
def parse_tagged_sentence(tagged_text, allowed_labels=None):
    if allowed_labels is None:
        allowed_labels = set(LABELS)
    else:
        allowed_labels = set(allowed_labels)

    pattern = re.compile(r"<([^:<>]+):([A-Z]{2})>")
    entities = []
    plain_text_parts = []
    last_idx = 0
    current_len = 0

    for match in pattern.finditer(tagged_text):
        start_tag, end_tag = match.span()
        entity_text = match.group(1)
        entity_label = match.group(2)

        normal_text = tagged_text[last_idx:start_tag]
        plain_text_parts.append(normal_text)
        current_len += len(normal_text)

        ent_start = current_len
        plain_text_parts.append(entity_text)
        current_len += len(entity_text)
        ent_end = current_len

        if entity_label in allowed_labels:
            entities.append({
                "start": ent_start,
                "end": ent_end,
                "label": entity_label,
                "text": entity_text
            })

        last_idx = end_tag

    remain_text = tagged_text[last_idx:]
    plain_text_parts.append(remain_text)

    plain_text = "".join(plain_text_parts)

    return {
        "text": plain_text,
        "entities": entities
    }
    # =========================================================
# 2. KLUE NER 로드
# =========================================================
dataset = load_dataset("klue", "ner")
print(dataset)
print(dataset["train"][0].keys())
print(dataset["train"][0])
# =========================================================
# 3. tokens + ner_tags -> entities 변환
# KLUE 구조 대응용
# =========================================================
def convert_klue_example(example, dataset_split):
    """
    KLUE NER 예제를
    {
      "text": ...,
      "labels": [...],
      "entities": [{"start":..., "end":..., "label":..., "text":...}]
    }
    형태로 변환
    """

    # case 1) sentence 안에 태그가 들어있는 경우
    sentence = example.get("sentence", None)
    if isinstance(sentence, str) and "<" in sentence and ">" in sentence and ":" in sentence:
        parsed = parse_tagged_sentence(sentence, allowed_labels=LABELS)
        return {
            "text": parsed["text"],
            "labels": LABELS,
            "entities": parsed["entities"]
        }

    # case 2) tokens, ner_tags 기반 처리
    tokens = example["tokens"]
    ner_tags = example["ner_tags"]

    # int -> str
    if len(ner_tags) > 0 and isinstance(ner_tags[0], int):
        ner_feature = dataset_split.features["ner_tags"].feature
        ner_tags = [ner_feature.int2str(x) for x in ner_tags]

    # sentence가 있으면 sentence 우선 사용
    if isinstance(sentence, str) and len(sentence) > 0:
        text = sentence
    else:
        # sentence가 없으면 tokens를 이어붙임
        text = "".join(tokens)

    entities = []
    i = 0
    cursor = 0

    # tokens를 text에서 순서대로 찾으면서 문자 span 매핑
    token_char_spans = []
    for tok in tokens:
        pos = text.find(tok, cursor)
        if pos == -1:
            pos = text.find(tok)
            if pos == -1:
                token_char_spans.append((None, None))
                continue
        token_char_spans.append((pos, pos + len(tok)))
        cursor = pos + len(tok)

    while i < len(tokens):
        tag = ner_tags[i]

        if tag == "O":
            i += 1
            continue

        if tag.startswith("B-"):
            label = tag.split("-")[-1]
            start_token_idx = i
            end_token_idx = i

            j = i + 1
            while j < len(tokens) and ner_tags[j] == f"I-{label}":
                end_token_idx = j
                j += 1

            start_char = token_char_spans[start_token_idx][0]
            end_char = token_char_spans[end_token_idx][1]

            if start_char is not None and end_char is not None and label in LABELS:
                entity_text = text[start_char:end_char]
                entities.append({
                    "start": start_char,
                    "end": end_char,
                    "label": label,
                    "text": entity_text
                })

            i = j
        else:
            i += 1

    return {
        "text": text,
        "labels": LABELS,
        "entities": entities
    }
    # =========================================================
# 4. 전체 split 변환
# =========================================================
processed_data = {}

for split in dataset.keys():
    split_items = []
    for ex in tqdm(dataset[split], desc=f"processing {split}"):
        item = convert_klue_example(ex, dataset[split])
        split_items.append(item)
    processed_data[split] = split_items

print("train:", len(processed_data["train"]))
print("validation:", len(processed_data["validation"]))
print(json.dumps(processed_data["train"][0], ensure_ascii=False, indent=2))
# =========================================================
# 5. 전처리 검증
# =========================================================
def validate_sample(sample):
    text = sample["text"]
    ok = True

    for ent in sample["entities"]:
        s, e = ent["start"], ent["end"]
        extracted = text[s:e]
        if extracted != ent["text"]:
            print("[SPAN ERROR]")
            print("text:", text)
            print("entity:", ent)
            print("extracted:", extracted)
            ok = False
    return ok

def validate_dataset(data, n_check=100):
    idxs = list(range(len(data)))
    random.shuffle(idxs)
    idxs = idxs[:min(n_check, len(idxs))]

    valid = 0
    for idx in idxs:
        if validate_sample(data[idx]):
            valid += 1

    print(f"valid: {valid}/{len(idxs)}")

validate_dataset(processed_data["train"], 100)
validate_dataset(processed_data["validation"], 100)
# =========================================================
# 6. JSON 저장
# =========================================================
for split in processed_data:
    save_path = os.path.join(SAVE_DIR, f"{split}.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(processed_data[split], f, ensure_ascii=False, indent=2)
    print("saved:", save_path)
    # =========================================================
# 7. 토크나이저 로드
# GlobalPointer는 token span이 필요하므로
# offset_mapping을 사용해서 문자 span -> 토큰 span 변환
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("tokenizer loaded:", MODEL_NAME)
# =========================================================
# 8. 문자 span -> 토큰 span 변환 함수
# end는 문자 기준 exclusive
# offset_mapping도 end exclusive
# =========================================================
def char_to_token_span(offset_mapping, char_start, char_end):
    """
    offset_mapping: [(0,0), (0,1), (1,2), ...]
    char_start, char_end: 문자 단위 span, [start, end)
    return: (token_start, token_end) or (None, None)
    """

    token_start = None
    token_end = None

    for idx, (s, e) in enumerate(offset_mapping):
        if s == e:
            continue  # special token

        if token_start is None and s <= char_start < e:
            token_start = idx

        # char_end는 exclusive 이므로 char_end-1이 포함된 token을 찾음
        if s < char_end <= e:
            token_end = idx
            break

        if char_end - 1 >= s and char_end - 1 < e:
            token_end = idx
            break

    return token_start, token_end
    # =========================================================
# 9. GlobalPointer용 단일 샘플 인코딩
# output:
# {
#   input_ids,
#   attention_mask,
#   token_type_ids(optional),
#   labels: (num_labels, max_len, max_len)
# }
# =========================================================
def encode_globalpointer_sample(sample, tokenizer, max_length=128):
    text = sample["text"]
    entities = sample["entities"]

    encoded = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_offsets_mapping=True,
        return_tensors=None
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]
    offset_mapping = encoded["offset_mapping"]

    token_type_ids = encoded.get("token_type_ids", [0] * len(input_ids))

    num_labels = len(LABELS)
    labels = torch.zeros((num_labels, max_length, max_length), dtype=torch.long)

    valid_entities = []

    for ent in entities:
        char_start = ent["start"]
        char_end = ent["end"]
        ent_label = ent["label"]

        token_start, token_end = char_to_token_span(offset_mapping, char_start, char_end)

        if token_start is None or token_end is None:
            continue

        if token_start >= max_length or token_end >= max_length:
            continue

        label_id = label2id[ent_label]
        labels[label_id, token_start, token_end] = 1

        valid_entities.append({
            "text": ent["text"],
            "label": ent_label,
            "char_start": char_start,
            "char_end": char_end,
            "token_start": token_start,
            "token_end": token_end
        })

    return {
        "text": text,
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
        "labels": labels,
        "offset_mapping": offset_mapping,
        "entities": entities,
        "valid_entities": valid_entities
    }
    # =========================================================
# 10. 샘플 확인
# =========================================================
sample_encoded = encode_globalpointer_sample(
    processed_data["train"][0],
    tokenizer,
    max_length=MAX_LENGTH
)

print("text:", sample_encoded["text"])
print("input_ids shape:", sample_encoded["input_ids"].shape)
print("attention_mask shape:", sample_encoded["attention_mask"].shape)
print("labels shape:", sample_encoded["labels"].shape)
print("valid_entities:")
for x in sample_encoded["valid_entities"]:
    print(x)
    # =========================================================
# 11. Dataset 클래스
# =========================================================
class GlobalPointerNERDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        item = encode_globalpointer_sample(
            sample,
            tokenizer=self.tokenizer,
            max_length=self.max_length
        )

        return {
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"],
            "token_type_ids": item["token_type_ids"],
            "labels": item["labels"]
        }
        # =========================================================
# 12. train/valid dataset 생성
# =========================================================
train_dataset = GlobalPointerNERDataset(
    processed_data["train"],
    tokenizer,
    max_length=MAX_LENGTH
)

valid_dataset = GlobalPointerNERDataset(
    processed_data["validation"],
    tokenizer,
    max_length=MAX_LENGTH
)

print("train_dataset:", len(train_dataset))
print("valid_dataset:", len(valid_dataset))
# =========================================================
# 13. collate_fn
# =========================================================
def collate_fn(batch):
    input_ids = torch.stack([x["input_ids"] for x in batch], dim=0)
    attention_mask = torch.stack([x["attention_mask"] for x in batch], dim=0)
    token_type_ids = torch.stack([x["token_type_ids"] for x in batch], dim=0)
    labels = torch.stack([x["labels"] for x in batch], dim=0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
        "labels": labels
    }
    # =========================================================
# 14. dataloader 생성
# =========================================================
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn
)

batch = next(iter(train_loader))
print("input_ids:", batch["input_ids"].shape)        # (B, L)
print("attention_mask:", batch["attention_mask"].shape)
print("token_type_ids:", batch["token_type_ids"].shape)
print("labels:", batch["labels"].shape)              # (B, num_labels, L, L)
# =========================================================
# 15. 학습 전 저장용 pt 파일 만들기
# 필요하면 미리 tensor dataset으로 저장 가능
# =========================================================
def build_encoded_cache(data, tokenizer, max_length=128):
    encoded_items = []
    for sample in tqdm(data, desc="encoding cache"):
        item = encode_globalpointer_sample(sample, tokenizer, max_length=max_length)
        encoded_items.append({
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"],
            "token_type_ids": item["token_type_ids"],
            "labels": item["labels"]
        })
    return encoded_items

# 시간이 좀 걸릴 수 있음
train_cache = build_encoded_cache(processed_data["train"], tokenizer, MAX_LENGTH)
valid_cache = build_encoded_cache(processed_data["validation"], tokenizer, MAX_LENGTH)

torch.save(train_cache, os.path.join(SAVE_DIR, "train_cache.pt"))
torch.save(valid_cache, os.path.join(SAVE_DIR, "valid_cache.pt"))

print("cache saved")
class CachedGlobalPointerDataset(Dataset):
    def __init__(self, cache_path):
        self.data = torch.load(cache_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

cached_train_dataset = CachedGlobalPointerDataset(os.path.join(SAVE_DIR, "train_cache.pt"))
cached_valid_dataset = CachedGlobalPointerDataset(os.path.join(SAVE_DIR, "valid_cache.pt"))

cached_train_loader = DataLoader(
    cached_train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

batch = next(iter(cached_train_loader))
print(batch["labels"].shape)
print(batch["input_ids"].shape)
print(batch["labels"].shape)

SAVE_DIR: /content/klue_ner_globalpointer
LABELS: ['DT', 'LC', 'OG', 'PS', 'QT', 'TI']
DatasetDict({
    train: Dataset({
        features: ['sentence', 'tokens', 'ner_tags'],
        num_rows: 21008
    })
    validation: Dataset({
        features: ['sentence', 'tokens', 'ner_tags'],
        num_rows: 5000
    })
})
dict_keys(['sentence', 'tokens', 'ner_tags'])
{'sentence': '특히 <영동고속도로:LC> <강릉:LC> 방향 <문막휴게소:LC>에서 <만종분기점:LC>까지 <5㎞:QT> 구간에는 승용차 전용 임시 갓길차로제를 운영하기로 했다.', 'tokens': ['특', '히', ' ', '영', '동', '고', '속', '도', '로', ' ', '강', '릉', ' ', '방', '향', ' ', '문', '막', '휴', '게', '소', '에', '서', ' ', '만', '종', '분', '기', '점', '까', '지', ' ', '5', '㎞', ' ', '구', '간', '에', '는', ' ', '승', '용', '차', ' ', '전', '용', ' ', '임', '시', ' ', '갓', '길', '차', '로', '제', '를', ' ', '운', '영', '하', '기', '로', ' ', '했', '다', '.'], 'ner_tags': [12, 12, 12, 2, 3, 3, 3, 3, 3, 12, 2, 3, 12, 12, 12, 12, 2, 3, 3, 3, 3, 12, 12, 12, 2, 3, 3, 3, 3, 12, 12, 12, 8, 9, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,

processing validation: 100%|██████████| 5000/5000 [00:01<00:00, 2851.45it/s]


train: 21008
validation: 5000
{
  "text": "특히 영동고속도로 강릉 방향 문막휴게소에서 만종분기점까지 5㎞ 구간에는 승용차 전용 임시 갓길차로제를 운영하기로 했다.",
  "labels": [
    "DT",
    "LC",
    "OG",
    "PS",
    "QT",
    "TI"
  ],
  "entities": [
    {
      "start": 3,
      "end": 9,
      "label": "LC",
      "text": "영동고속도로"
    },
    {
      "start": 10,
      "end": 12,
      "label": "LC",
      "text": "강릉"
    },
    {
      "start": 16,
      "end": 21,
      "label": "LC",
      "text": "문막휴게소"
    },
    {
      "start": 24,
      "end": 29,
      "label": "LC",
      "text": "만종분기점"
    },
    {
      "start": 32,
      "end": 34,
      "label": "QT",
      "text": "5㎞"
    }
  ]
}
valid: 100/100
valid: 100/100
saved: /content/klue_ner_globalpointer/train.json
saved: /content/klue_ner_globalpointer/validation.json
tokenizer loaded: klue/roberta-base
text: 특히 영동고속도로 강릉 방향 문막휴게소에서 만종분기점까지 5㎞ 구간에는 승용차 전용 임시 갓길차로제를 운영하기로 했다.
input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
labels shape: tor

encoding cache:  71%|███████   | 14846/21008 [01:12<00:05, 1047.33it/s]